In [1]:
from datasets import load_dataset, load_from_disk
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer, DataCollatorCTCWithPadding
import soundfile as sf
import numpy as np
import torch

# 1) Load just the dirty_sa splits (streaming or cached)
from pathlib import Path
from datasets import load_dataset

BASE = Path("/home/BTECH_7TH_SEM/Desktop/NLP Datasets/peoples_speech_dirty_sa/dirty_sa")

def load_parquet_split(split_name):
    # find all parquet files under the split folder (recursively)
    files = sorted([str(p) for p in (BASE / split_name).rglob("*.parquet")])
    if not files:
        raise FileNotFoundError(f"No parquet files found for split {split_name} at {BASE/split_name}")
    # load as a HF dataset using the parquet loader
    ds = load_dataset("parquet", data_files=files, split="train")  # 'train' here just creates a Dataset
    return ds

train_ds = load_parquet_split("train")
valid_ds = load_parquet_split("validation")
test_ds  = load_parquet_split("test")

print("Train columns:", train_ds.column_names)
print("One example:\n", train_ds[0])

# 2) Choose a pretrained checkpoint
model_name = "facebook/wav2vec2-base-960h"   # or "facebook/wav2vec2-large-xlsr-53" or a w2v-bert variant
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

# 3) Preprocess function (resample to 16k, extract input_values and labels)
def prepare_batch(batch):
    # example: files might be in 'audio' field as { 'path': ..., 'array': ... } depending on dataset format
    audio = batch["audio"]  # dataset audio column is often a dict with 'path' or 'array'
    if isinstance(audio, dict) and "path" in audio:
        speech, sr = sf.read(audio["path"])
    else:
        speech, sr = audio["array"], audio["sampling_rate"]
    # resample if needed (use torchaudio in full code)
    if sr != 16000:
        import torchaudio
        speech = torchaudio.functional.resample(torch.tensor(speech).float(), orig_freq=sr, new_freq=16000).numpy()
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=False)
    with processor.as_target_processor():
        labels = processor(batch["transcript"]).input_ids
    batch["input_values"] = inputs.input_values[0]
    batch["labels"] = labels
    return batch

# Map preprocessing (use batched=True for speed, handle memory)
train_ds = train_ds.map(prepare_batch)
valid_ds = valid_ds.map(prepare_batch)

# 4) Data collator
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# 5) Training args + Trainer
training_args = TrainingArguments(
    output_dir="./wav2vec2-dirty_sa",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    logging_steps=200,
    num_train_epochs=3,
    fp16=True,
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=processor.feature_extractor,   # processor contains feature_extractor + tokenizer
    data_collator=data_collator
)

trainer.train()


/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/NLP/nlp-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 